In [16]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import datetime
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

In [17]:
raw_labels = pd.read_csv('/Users/eric/repos/aud/data/day_labels.csv')
subject_info = pd.read_csv('/Users/eric/repos/aud/data/subject_info.csv')
train_widths = [15,30,45,60,75,90]
for width in train_widths:
    features = raw_labels.copy(deep=True).reset_index(drop=True)
    counter = 0
    missing_subids = []
    print("Width = {}".format(str(width)))
    # Create additional label columns for shifted days
    features['lapse_d0'] = features['lapse']
    day_shift = [1,2,3,4,5,6,7]
    for delta in day_shift:
        col_name = 'lapse_d'+str(delta)
        features[col_name]=features.groupby('subid')['lapse'].shift(-delta)

    # Pre-allocate new feature columns
    # Latest EMA value
    ema_nums = [2,3,4,5,6,7,8,9,10]
    for i in ema_nums:
        added_str = 'latest_ema_'+str(i)
        features[added_str] = np.nan
    # Short-run mean EMA value (average of up to 3 most recent previous EMAs)
    for i in ema_nums:
        added_str = 'srm_ema_'+str(i)
        features[added_str] = np.nan
    # Long-run mean EMA value (average of all EMAs prior to the current day)
    for i in ema_nums:
        added_str = 'lrm_ema_'+str(i)
        features[added_str] = np.nan
    # Recent lapse
    features["recent_lapse_1"] = np.nan
    features["recent_lapse_3"] = np.nan
    features["recent_lapse_5"] = np.nan

    # Additional labels for prediction
    # Lapse-within-window
    features['lapse_w0'] = features['lapse']
    features['lapse_w1'] = np.nan
    features['lapse_w3'] = np.nan
    features['lapse_w7'] = np.nan

    # Loop through the dataset and add feature values
    #for idx,row in features.iloc[1:5].iterrows():
    for idx,row in features.iterrows():
        # General information
        subid = row.subid
        day = row.day
        lb = day - width
        # First/last EMA info
        sub_info = subject_info.query("subid==@subid")
        if sub_info.shape[0] > 1:
            print("Error in subject info, too many lines")
            exit
        else:
            last_ema = sub_info.iloc[0].last_morning_ema_day
            first_ema = sub_info.iloc[0].first_morning_ema_day

        current_df = features.query("subid==@subid & day<=@day & day>@lb").sort_values(by='day',ascending=False).copy(deep=True)
        current_df_nonan = current_df.dropna(subset=['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10'])
        past_df = features.query("subid==@subid & day < @day").sort_values(by='day',ascending=False).copy(deep=True)
        past_df_nonan = past_df.dropna(subset=['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10'])

        # Latest EMA values
        # Empty data case
        if current_df_nonan.shape[0]==0:
            # No need to do anything to set this row to NaN since it is preallocated as NaN
            if day>0:
                counter+=1
                if subid not in missing_subids:
                    missing_subids.append(subid)
            #pass
        # Non-empty data case
        else:
            recent_vals=current_df_nonan.iloc[0]
            # Put these values into the dataframe in the correct row
            raw_vals = recent_vals[['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']].tolist()
            features.loc[idx,['latest_ema_2','latest_ema_3','latest_ema_4','latest_ema_5','latest_ema_6','latest_ema_7','latest_ema_8','latest_ema_9','latest_ema_10']]=raw_vals

        # Short-run mean EMA value (average of the last 3 non-NaN responses, does include today)
        # Empty data case
        if current_df_nonan.shape[0]==0:
            # No need to do anything to set this row to NaN since it is preallocated as NaN
            pass
        # Non-empty data case
        else:
            # Take up to 3 datapoints (as available)
            end_idx = min(current_df_nonan.shape[0],3)
            #recent_vals=past_df_nonan.iloc[0:end_idx]
            recent_vals=current_df_nonan.iloc[0:end_idx]
            # Put these values into the dataframe in the correct row
            raw_vals = recent_vals[['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']].mean().tolist()
            features.loc[idx,['srm_ema_2','srm_ema_3','srm_ema_4','srm_ema_5','srm_ema_6','srm_ema_7','srm_ema_8','srm_ema_9','srm_ema_10']]=raw_vals

        # Long-run mean EMA value (average of all non-NaN responses, does include today)
        # Empty data case
        if current_df_nonan.shape[0]==0:
            # No need to do anything to set this row to NaN since it is preallocated as NaN
            pass
        # Non-empty data case
        else:
            long_end_idx = current_df_nonan.shape[0]
            #recent_vals=past_df_nonan
            recent_vals=current_df_nonan.iloc[0:long_end_idx]
            # Put these values into the dataframe in the correct row
            raw_vals = recent_vals[['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10']].mean().tolist()
            features.loc[idx,['lrm_ema_2','lrm_ema_3','lrm_ema_4','lrm_ema_5','lrm_ema_6','lrm_ema_7','lrm_ema_8','lrm_ema_9','lrm_ema_10']]=raw_vals

        # Day of week dummies
        # Leaving this in for now due to odd behavior in tree-based models
        #features.drop(labels='day_of_week_num_0',axis=1,inplace=True)

        # Recent lapse (lapse occurring within the last 1,3,5 days)
        end_idx = min(1,past_df.shape[0])
        if end_idx >0:
            recent_vals = past_df.iloc[0:end_idx].lapse.to_numpy()
            recent_lapse = np.any(recent_vals)
        else:
            recent_lapse = False
        features.loc[idx,'recent_lapse_1']=recent_lapse
        
        end_idx = min(3,past_df.shape[0])
        if end_idx >0:
            recent_vals = past_df.iloc[0:end_idx].lapse.to_numpy()
            recent_lapse = np.any(recent_vals)
        else:
            recent_lapse = False
        features.loc[idx,'recent_lapse_3']=recent_lapse

        end_idx = min(5,past_df.shape[0])
        if end_idx >0:
            recent_vals = past_df.iloc[0:end_idx].lapse.to_numpy()
            recent_lapse = np.any(recent_vals)
        else:
            recent_lapse = False
        features.loc[idx,'recent_lapse_5']=recent_lapse

        # Labels
        # Create shifted label columns
        w1_vec = [row.lapse_d0,row.lapse_d1]
        if any(np.isnan(w1_vec)) or (day+1)>last_ema:
            val = np.nan
        else:
            val = int(np.any(w1_vec))
        features.loc[idx,'lapse_w1'] = val

        w3_vec = [row.lapse_d0,row.lapse_d1,row.lapse_d2,row.lapse_d3]
        if any(np.isnan(w3_vec)) or (day+3)>last_ema:
            val = np.nan
        else:
            val = int(np.any(w3_vec))
        features.loc[idx,'lapse_w3'] = val

        w7_vec = [row.lapse_d0,row.lapse_d1,row.lapse_d2,row.lapse_d3,row.lapse_d4,row.lapse_d5,row.lapse_d6,row.lapse_d7]
        if any(np.isnan(w7_vec)) or (day+7)>last_ema:
            val = np.nan
        else:
            val = int(np.any(w7_vec))
        features.loc[idx,'lapse_w7'] = val

    # Drop columns
    features.drop(labels=['lapse_before_mema','lapse_after_mema','mema_arrival','lapse_start','lapse_end'],axis=1,inplace=True)

    # Filter out the subjects with less than 25% of days having morning EMAs

    included_subjects = subject_info.query("responsiveness >= 0.25").subid.unique()
    dropped_subject = subject_info.query("responsiveness < 0.25").subid.unique()
    print(dropped_subject)
    #included_subjects = subject_info.query("mema_count >= 60").subid.to_numpy()
    export_df = features.query("subid in @included_subjects").copy(deep=True)

    # Low EMA drops
    print(features.shape)
    print(export_df.shape)

    # Drop lines after the last EMA day or before the first
    for idx,row in subject_info.iterrows():
        subid = row.subid
        mema_count = row.mema_count
        last_ema = row.last_morning_ema_day
        first_ema = row.first_morning_ema_day
        drop_sub_df = export_df.query("subid == @subid & (day > @last_ema | day < @first_ema)")
        idxs = drop_sub_df.index.to_numpy()
        export_df.drop(labels=idxs,axis=0,inplace=True)

    print(export_df.shape)

    # Drop NA entries in the EMA? Confirm that this is only a few cases
    idxs1 = export_df.loc[export_df.latest_ema_2.isnull()].index.to_numpy()
    print(idxs1)
    display(export_df.loc[idxs1])
    export_df.drop(labels=idxs1,axis=0,inplace=True)
    print(export_df.shape)
    ml_file_name = 'ml_labels_{}.csv'.format(str(width))
    # Column dropping
    export_df.drop(labels = ['ema_2','ema_3','ema_4','ema_5','ema_6','ema_7','ema_8','ema_9','ema_10','lapse',
                            'lapse_d0', 'lapse_d1', 'lapse_d2', 'lapse_d3', 'lapse_d4', 'lapse_d5', 'lapse_d6', 'lapse_d7'],axis=1,inplace=True)
    export_df.to_csv(ml_file_name,index=False)
    #print(counter)
    #print(missing_subids)

Width = 15
[ 80 193 232]
(13590, 61)
(13320, 61)
(12572, 61)
[ 4129  9714 10516 10517 10518 10519 10520 10521 10522 10523 10524 10525
 10526]


,subid,day,ema_2,ema_3,ema_4,ema_5,ema_6,ema_7,ema_8,ema_9,ema_10,lapse,day_of_week_num_0,day_of_week_num_1,day_of_week_num_2,day_of_week_num_3,day_of_week_num_4,day_of_week_num_5,day_of_week_num_6,lapse_d0,lapse_d1,lapse_d2,lapse_d3,lapse_d4,lapse_d5,lapse_d6,lapse_d7,latest_ema_2,latest_ema_3,latest_ema_4,latest_ema_5,latest_ema_6,latest_ema_7,latest_ema_8,latest_ema_9,latest_ema_10,srm_ema_2,srm_ema_3,srm_ema_4,srm_ema_5,srm_ema_6,srm_ema_7,srm_ema_8,srm_ema_9,srm_ema_10,lrm_ema_2,lrm_ema_3,lrm_ema_4,lrm_ema_5,lrm_ema_6,lrm_ema_7,lrm_ema_8,lrm_ema_9,lrm_ema_10,recent_lapse_1,recent_lapse_3,recent_lapse_5,lapse_w0,lapse_w1,lapse_w3,lapse_w7
4129,66,79,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0.0,0.0,0.0
9714,191,84,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,1,0,0,0,0,0.0,1.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0.0,1.0,NaN
10516,207,76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0.0,0.0,0.0
10517,207,77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0.0,0.0,0.0
10518,207,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0.0,0.0,0.0
10519,207,79,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0.0,0.0,0.0
10520,207,80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0.0,0.0,0.0
10521,207,81,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,1,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0.0,0.0,0.0
10522,207,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0.0,0.0,0.0
10523,207,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,0,0.0,0.0,NaN


(12559, 61)
Width = 30
[ 80 193 232]
(13590, 61)
(13320, 61)
(12572, 61)
[]


,subid,day,ema_2,ema_3,ema_4,ema_5,ema_6,ema_7,ema_8,ema_9,ema_10,lapse,day_of_week_num_0,day_of_week_num_1,day_of_week_num_2,day_of_week_num_3,day_of_week_num_4,day_of_week_num_5,day_of_week_num_6,lapse_d0,lapse_d1,lapse_d2,lapse_d3,lapse_d4,lapse_d5,lapse_d6,lapse_d7,latest_ema_2,latest_ema_3,latest_ema_4,latest_ema_5,latest_ema_6,latest_ema_7,latest_ema_8,latest_ema_9,latest_ema_10,srm_ema_2,srm_ema_3,srm_ema_4,srm_ema_5,srm_ema_6,srm_ema_7,srm_ema_8,srm_ema_9,srm_ema_10,lrm_ema_2,lrm_ema_3,lrm_ema_4,lrm_ema_5,lrm_ema_6,lrm_ema_7,lrm_ema_8,lrm_ema_9,lrm_ema_10,recent_lapse_1,recent_lapse_3,recent_lapse_5,lapse_w0,lapse_w1,lapse_w3,lapse_w7


(12572, 61)
Width = 45
[ 80 193 232]
(13590, 61)
(13320, 61)
(12572, 61)
[]


...

(12572, 61)
Width = 75
[ 80 193 232]
(13590, 61)
(13320, 61)
(12572, 61)
[]


,subid,day,ema_2,ema_3,ema_4,ema_5,ema_6,ema_7,ema_8,ema_9,ema_10,lapse,day_of_week_num_0,day_of_week_num_1,day_of_week_num_2,day_of_week_num_3,day_of_week_num_4,day_of_week_num_5,day_of_week_num_6,lapse_d0,lapse_d1,lapse_d2,lapse_d3,lapse_d4,lapse_d5,lapse_d6,lapse_d7,latest_ema_2,latest_ema_3,latest_ema_4,latest_ema_5,latest_ema_6,latest_ema_7,latest_ema_8,latest_ema_9,latest_ema_10,srm_ema_2,srm_ema_3,srm_ema_4,srm_ema_5,srm_ema_6,srm_ema_7,srm_ema_8,srm_ema_9,srm_ema_10,lrm_ema_2,lrm_ema_3,lrm_ema_4,lrm_ema_5,lrm_ema_6,lrm_ema_7,lrm_ema_8,lrm_ema_9,lrm_ema_10,recent_lapse_1,recent_lapse_3,recent_lapse_5,lapse_w0,lapse_w1,lapse_w3,lapse_w7


(12572, 61)
Width = 90
[ 80 193 232]
(13590, 61)
(13320, 61)
(12572, 61)
[]


,subid,day,ema_2,ema_3,ema_4,ema_5,ema_6,ema_7,ema_8,ema_9,ema_10,lapse,day_of_week_num_0,day_of_week_num_1,day_of_week_num_2,day_of_week_num_3,day_of_week_num_4,day_of_week_num_5,day_of_week_num_6,lapse_d0,lapse_d1,lapse_d2,lapse_d3,lapse_d4,lapse_d5,lapse_d6,lapse_d7,latest_ema_2,latest_ema_3,latest_ema_4,latest_ema_5,latest_ema_6,latest_ema_7,latest_ema_8,latest_ema_9,latest_ema_10,srm_ema_2,srm_ema_3,srm_ema_4,srm_ema_5,srm_ema_6,srm_ema_7,srm_ema_8,srm_ema_9,srm_ema_10,lrm_ema_2,lrm_ema_3,lrm_ema_4,lrm_ema_5,lrm_ema_6,lrm_ema_7,lrm_ema_8,lrm_ema_9,lrm_ema_10,recent_lapse_1,recent_lapse_3,recent_lapse_5,lapse_w0,lapse_w1,lapse_w3,lapse_w7


(12572, 61)


In [18]:
df_list = []
for width in train_widths[:-1]:
    file_name = 'ml_labels_{}.csv'.format(str(width))
    df = pd.read_csv(file_name)
    df['train_width']=width
    df_list.append(df)

limited_df = pd.concat(df_list,ignore_index=True)
limited_df.groupby(by='train_width').head()
limited_df.to_csv('/Users/eric/repos/aud/data/ml_labels_limited_width.csv',index=False)

In [19]:
file_name = 'ml_labels_90.csv'.format(str(width))
df = pd.read_csv(file_name)
df['train_width']=90
df.head()
df.to_csv('/Users/eric/repos/aud/data/ml_labels_full_width.csv',index=False)